# 08 — Error Analysis & Final Discussion

**Inputs:**
- `data/processed/X_features.csv`
- `data/processed/y_target.csv`
- `models/best_model.pkl`
- `models/scaler.pkl`

**What this notebook does:**
1. Loads model and test data
2. Identifies exactly which test students were misclassified
3. Examines the *features* of misclassified students
4. Produces final summary tables for the thesis
5. Writes the complete discussion + limitations + future work
   sections

**This notebook is the wrap-up of the entire ML pipeline.**

Notebooks 01–07 built the pipeline. This notebook interprets it.

In [1]:
# Step 1: Load model, scaler, features, target.

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

# Load
X = pd.read_csv("../data/processed/X_features.csv")
y = pd.read_csv("../data/processed/y_target.csv").squeeze("columns")

# Recreate split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Load model and scaler
scaler = joblib.load("../models/scaler.pkl")
model  = joblib.load("../models/best_model.pkl")

# Scale test
X_test_scaled = scaler.transform(X_test)

# Predict
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)

print("Loaded.")
print("Test shape:", X_test.shape)
print("Predictions shape:", y_pred.shape)

Loaded.
Test shape: (8, 37)
Predictions shape: (8,)


## Step 2 — Error Analysis Table

For each of the 8 test students we record:

- **True label** — from the survey
- **Predicted label** — from the model
- **Correct?** — True / False
- **Confidence** — the model's predicted probability for the chosen class
- **True-class probability** — how likely the model thought the TRUE class was

When true-class probability is very low but confidence is high,
the model is confidently wrong — the most dangerous failure mode.

With n=8 test samples, each row represents 12.5% of accuracy.
We should not over-interpret individual errors, but we can look for
systematic patterns (e.g., "all errors were Moderate vs Low").

In [2]:
# Step 2: Build the error analysis table.

# Assemble a summary table
errors_df = pd.DataFrame({
    "true_label": y_test.values,
    "predicted": y_pred,
    "correct": (y_test.values == y_pred),
    "confidence": y_proba.max(axis=1).round(3),
})

# Add probability the model assigned to the TRUE class
class_to_index = {cls: i for i, cls in enumerate(model.classes_)}
errors_df["true_class_prob"] = [
    y_proba[i, class_to_index[y_test.iloc[i]]].round(3)
    for i in range(len(y_test))
]

print("Error Analysis Table (test set, n=8):")
print(errors_df.to_string(index=True))
print()

# Confusion summary
n_correct = errors_df["correct"].sum()
n_wrong   = len(errors_df) - n_correct
print(f"Correct: {n_correct}/{len(errors_df)}  ({n_correct/len(errors_df)*100:.1f}%)")
print(f"Wrong  : {n_wrong}/{len(errors_df)}  ({n_wrong/len(errors_df)*100:.1f}%)")

Error Analysis Table (test set, n=8):
  true_label predicted  correct  confidence  true_class_prob
0        Low  Moderate    False       1.000              0.0
1   Moderate       Low    False       1.000              0.0
2       High      High     True       1.000              1.0
3        Low  Moderate    False       1.000              0.0
4       High  Moderate    False       1.000              0.0
5   Moderate      High    False       1.000              0.0
6        Low       Low     True       1.000              1.0
7   Moderate      High    False       0.667              0.0

Correct: 2/8  (25.0%)
Wrong  : 6/8  (75.0%)


## Step 3 — Error Pattern Analysis

We characterize the errors to see if they are random or systematic.

Specifically we check:
- Are the errors between *adjacent* stress classes (Low↔Moderate, Moderate↔High)?
- Or between *distant* classes (Low↔High)?

If errors are mostly adjacent, the model is making "close" mistakes — a
less severe failure mode. If errors are distant, the model's class
boundaries are fundamentally wrong.

We also check the model's confidence calibration: a well-calibrated
model should be less confident when it is wrong.

In [4]:
# Step 3: Characterize the error patterns.

# Define class ordering (Low < Moderate < High)
order = {"Low": 0, "Moderate": 1, "High": 2}

# Compute "distance" of each error (in class-rank units)
distances = []
for _, row in errors_df.iterrows():
    if not row["correct"]:
        d = abs(order[row["true_label"]] - order[row["predicted"]])
        distances.append(d)

import numpy as np
distances = np.array(distances)

print("Error distances (0=correct, 1=adjacent, 2=opposite):")
print(distances)
print()
print(f"Total errors              : {len(distances)}")
print(f"Adjacent (|Δ|=1)          : {(distances == 1).sum()}")
print(f"Opposite (|Δ|=2)          : {(distances == 2).sum()}")
print()

# Confidence calibration
print("Model confidence when:")
print(f"  Correct predictions: mean = {errors_df.loc[errors_df['correct'], 'confidence'].mean():.3f}")
print(f"  Wrong predictions  : mean = {errors_df.loc[~errors_df['correct'], 'confidence'].mean():.3f}")
print()
print("Expected behavior of a well-calibrated model:")
print("  → Confident when correct, LESS confident when wrong")
print("If both numbers are high, model is OVERCONFIDENT.")

Error distances (0=correct, 1=adjacent, 2=opposite):
[1 1 1 1 1 1]

Total errors              : 6
Adjacent (|Δ|=1)          : 6
Opposite (|Δ|=2)          : 0

Model confidence when:
  Correct predictions: mean = 1.000
  Wrong predictions  : mean = 0.945

Expected behavior of a well-calibrated model:
  → Confident when correct, LESS confident when wrong
If both numbers are high, model is OVERCONFIDENT.


## Step 4 — Appendix Experiment: Constrained Decision Tree

The default Decision Tree overfit severely (test accuracy 0.25).
A common remedy is to **constrain the tree's complexity** so it cannot
memorize the training set.

We train a second Decision Tree with:
- `max_depth = 3` (instead of unlimited)
- `min_samples_leaf = 3` (each leaf must contain at least 3 samples)

If this simpler model performs *better* on the test set, it confirms
that the original tree's failure was due to overfitting — a
methodological lesson.

**Important:** This experiment is documented as an appendix analysis.
It does NOT replace the pre-selected model from Notebook 05. The
original selection is preserved.

In [6]:
# Step 4: Train a constrained Decision Tree and evaluate.

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# --- Original (already loaded) model: evaluate on test set ---
y_pred_original = model.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred_original)
test_f1  = f1_score(y_test, y_pred_original, average="macro")

# --- Constrained tree: train fresh ---
X_train_scaled = scaler.transform(X_train)   # make sure it's scaled

tree_simple = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=3,
    random_state=42
)
tree_simple.fit(X_train_scaled, y_train)

y_pred_simple = tree_simple.predict(X_test_scaled)

acc_simple = accuracy_score(y_test, y_pred_simple)
f1_simple  = f1_score(y_test, y_pred_simple, average="macro")

# --- Compare ---
print("Original Decision Tree (max_depth=5, min_samples_leaf=1):")
print(f"  Test accuracy : {test_acc:.3f}")
print(f"  Test F1 macro : {test_f1:.3f}")
print()

print("Constrained Decision Tree (max_depth=3, min_samples_leaf=3):")
print(f"  Test accuracy : {acc_simple:.3f}")
print(f"  Test F1 macro : {f1_simple:.3f}")
print()

# --- Confusion matrix for the constrained tree ---
cm_simple = confusion_matrix(y_test, y_pred_simple, labels=["Low", "Moderate", "High"])
cm_simple_df = pd.DataFrame(cm_simple,
                             index=["True Low", "True Mod", "True High"],
                             columns=["Pred Low", "Pred Mod", "Pred High"])
print("Confusion Matrix — Constrained Tree:")
print(cm_simple_df)

Original Decision Tree (max_depth=5, min_samples_leaf=1):
  Test accuracy : 0.250
  Test F1 macro : 0.267

Constrained Decision Tree (max_depth=3, min_samples_leaf=3):
  Test accuracy : 0.375
  Test F1 macro : 0.357

Confusion Matrix — Constrained Tree:
           Pred Low  Pred Mod  Pred High
True Low          2         1          0
True Mod          2         0          1
True High         0         1          1


## Step 5 — Final Discussion

### Summary of the ML Pipeline

The complete machine learning pipeline for predicting academic stress
among undergraduate students in Nepal was implemented and evaluated:

1. **Data collection**: 39 survey responses via Google Forms
2. **Data cleaning**: standardized column names, verified no missing values
3. **Exploratory analysis**: distributions, target construction
4. **Target variable**: total academic stress score (10–50), tertile-split
   into Low (n=15), Moderate (n=13), High (n=11)
5. **Feature engineering**: 37 features from 21 raw variables
   (ordinal encoding + one-hot encoding)
6. **Model training**: 7 classifiers evaluated with 5-fold stratified CV
7. **Best CV model**: Decision Tree (F1 macro = 0.562 ± 0.262)
8. **Test evaluation**: F1 macro = 0.267 (accuracy 0.250)
9. **Explainability**: SHAP identified 5 primary features but
   showed unstable, sometimes counterintuitive effects
10. **Error analysis**: All 6 test errors were "adjacent" classes
    (no Low↔High confusions); the model was severely overconfident
11. **Appendix experiment**: A constrained tree (max_depth=3) improved
    test F1 to 0.357, confirming the overfitting diagnosis

---

### Primary Findings

**1. The dataset is too small for reliable ML modeling.**
With n=39 and p=37, the feature-to-sample ratio (p/n ≈ 1) is far
outside the healthy range for supervised learning. Cross-validation
scores (0.562) overestimate true generalization (0.267).

**2. Decision Trees overfit severely on small samples.**
The unconstrained tree memorized the training set, producing
100%-confident wrong predictions on test data. Constraining the
tree (max_depth=3, min_samples_leaf=3) improved generalization,
though accuracy remained at baseline level.

**3. Errors are structurally benign.**
All 6 test errors were between adjacent stress classes. No
Low↔High confusions occurred. This suggests that the model's
learned class *ordering* is meaningful, even when the exact class
is wrong.

**4. SHAP feature attribution is unreliable at this sample size.**
SHAP identified `academic_life_satisfaction`, `social_support`,
`gender_Male`, and `academic_performance` as important, but the
directions of some effects were inconsistent with psychological
theory. This is consistent with overfitting rather than a genuine
signal.

---

### Limitations

- **Sample size**: n=39 is far below the ~300–500 typically
  required for reliable classification.
- **Class imbalance in the test set**: with only 2–3 samples per
  class, per-class metrics are noisy.
- **Self-reported data**: survey responses may be affected by
  social desirability, recall bias, and misinterpretation.
- **Single-site sample**: primarily Tribhuvan University and BCA
  students; not representative of Nepal's undergraduate population.
- **No external validation**: results are from one 8-sample
  holdout; no cross-institution replication.
- **Target construction**: the stress score uses equal weighting
  of all 10 questions; alternative weighting schemes could yield
  different targets.

---

### Future Work

1. **Extend data collection to 300+ responses.** This is the single
   most important next step; it resolves the primary limitation.

2. **Stratify by region, university type, and program.** Ensures
   broad representation beyond Tribhuvan University / BCA students.

3. **Investigate alternative target definitions.** Continuous
   regression on `stress_score` and ordinal models are worth
   comparing against the tertile classification.

4. **Test feature reduction methods.** With n=300+, feature selection
   (mutual information, recursive feature elimination) may improve
   generalization.

5. **Explore interaction effects.** Domain-informed feature
   engineering could reveal combinations (e.g., financial stress ×
   part-time job) that carry meaningful predictive signal.

6. **Validate the pipeline with an external dataset.** Cross-
   institution replication would strengthen external validity.

7. **Deploy a simple web application** for future data collection
   and stress self-assessment, using the trained model once
   sample size is sufficient.

---

### Conclusion

This project demonstrates the **complete ML pipeline** for academic
stress prediction — from questionnaire design through explainability
analysis — but also documents an important negative result: with
39 responses, machine learning models do **not** generalize reliably,
regardless of algorithm choice. The work provides a reproducible
framework that will yield meaningful findings once the sample
reaches the required size.